# Experiment 05 — Client Update Geometry under Proximal Regularization

This notebook studies **why** changing the FedProx coefficient changes optimization under severe federated heterogeneity. Instead of introducing another algorithm, it measures what each client actually does before aggregation.

For client (k) at communication round (t),

\[
\Delta_k^{(t)} = w_k^{(t)} - w_t,
\]

where (w_t) is the round-start global model and (w_k^{(t)}) is the locally trained model.

### Main findings (single seed)

- Increasing μ substantially reduced update magnitude. Across all clients and rounds, mean magnitude fell by **18.7%** for μ=0.1 and **49.6%** for μ=1 relative to FedAvg.
- Moderate regularization (μ=0.1) reached 85% accuracy in **10 rounds**, compared with **12 rounds** for FedAvg, and achieved the best late-round mean accuracy (**87.07%**).
- Strong regularization (μ=1) produced the smallest updates but worse convergence, showing that smaller updates are not automatically better updates.
- Pairwise direction changed much less consistently than magnitude. The evidence does **not** support a simple claim that FedProx uniformly aligns clients.
- Ordinary aggregate alignment was mechanically positive because each client helped create its own reference direction. A leave-one-out reconstruction exposed substantially more peer conflict, especially for client 3.

All conclusions are descriptive associations from one seed; they are not statistical or causal claims about FedProx in general.

## 1. Research question and hypotheses

**Research question**

> How does proximal regularization affect client-update magnitude and directional alignment under strong non-IID data?

We test three hypotheses:

- **H1:** Increasing μ reduces \(\lVert\Delta_k\rVert_2\) because FedProx penalizes movement away from (w_t).
- **H2:** Moderate μ may improve directional agreement between clients.
- **H3:** Excessive μ may constrain useful local learning so strongly that convergence becomes slower or worse.

These hypotheses deliberately separate **how far** a client moves from **which direction** it moves.

## 2. Geometry diagnostics

### Update magnitude

\[
M_k^{(t)} = \lVert\Delta_k^{(t)}\rVert_2.
\]

This measures how far client (k) moves from the round-start global model.

### Pairwise update alignment

\[
A_{ij}^{(t)} =
\cos(\Delta_i^{(t)},\Delta_j^{(t)}).
\]

Positive values indicate similar directions, values near zero indicate near-orthogonality, and negative values indicate conflict.

### Aggregate update alignment

FedAvg's sample-size-weighted update is

\[
\bar\Delta^{(t)} = \sum_k p_k\Delta_k^{(t)},
\qquad
p_k=\frac{n_k}{\sum_j n_j}.
\]

The ordinary aggregate alignment is

\[
G_k^{(t)}=\cos(\Delta_k^{(t)},\bar\Delta^{(t)}).
\]

Because \(\bar\Delta\) contains \(p_k\Delta_k\), this diagnostic includes a positive self-term:

\[
\Delta_k^\top\bar\Delta
=p_k\lVert\Delta_k\rVert^2
+\sum_{j\ne k}p_j\Delta_k^\top\Delta_j.
\]

We therefore also report a peer-only diagnostic:

\[
G_{-k}^{(t)}=
\cos\left(
\Delta_k^{(t)},
\sum_{j\ne k}p_j\Delta_j^{(t)}
\right).
\]

Leave-one-out alignment measures agreement with the other clients; it is not a performance-contribution score and does not prove that a client is harmful.

## 3. Controlled experimental design

| Component | Setting |
|---|---|
| Dataset | MNIST |
| Model | SimpleMLP, 784 → 128 → 10 |
| Clients | 5, full participation |
| Partition | Saved Dirichlet α=0.1 partition |
| Heterogeneity | Label concentration and quantity imbalance |
| Initialization | Shared saved checkpoint (w_0) |
| Initial test loss / accuracy | 2.313339 / 12.67% |
| Local epochs | 5 |
| Communication rounds | 20 |
| Batch size | 64 |
| Optimizer | SGD, learning rate 0.01 |
| Proximal coefficients | μ ∈ {0, 0.1, 1} |
| Seed | 42 |

μ=0 is the FedAvg-equivalent control, μ=0.1 is moderate regularization, and μ=1 is the deliberately strong condition.

## 4. Load saved results

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")


def find_result_dir():
    '''Locate results when opened from the repository or deliverable folder.'''
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]

    for candidate in candidates:
        result_dir = candidate / "results" / "client_update_geometry"
        if result_dir.is_dir():
            return result_dir

    raise FileNotFoundError(
        "Could not find results/client_update_geometry. "
        "Run this notebook from the repository or keep the supplied results folder."
    )


RESULT_DIR = find_result_dir()
ALPHA_DIR = RESULT_DIR / "alpha_0p1"
FIGURE_DIR = ALPHA_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Results:", RESULT_DIR)

In [ ]:
magnitude_df = pd.read_csv(ALPHA_DIR / "magnitude_E5_T20_seed42.csv")
pairwise_df = pd.read_csv(ALPHA_DIR / "pairwise_E5_T20_seed42.csv")
aggregate_alignment_df = pd.read_csv(
    ALPHA_DIR / "aggregate_alignment_E5_T20_seed42.csv"
)
performance_df = pd.read_csv(ALPHA_DIR / "performance_E5_T20_seed42.csv")
label_distribution_df = pd.read_csv(ALPHA_DIR / "client_label_distribution.csv")
scaling_performance_df = pd.read_csv(
    ALPHA_DIR / "client3_scaling_performance_E5_T20_seed42.csv"
)

expected_rows = {
    "magnitude": 3 * 20 * 5,
    "pairwise": 3 * 20 * 10,
    "aggregate alignment": 3 * 20 * 5,
    "performance": 3 * 20,
}
actual_rows = {
    "magnitude": len(magnitude_df),
    "pairwise": len(pairwise_df),
    "aggregate alignment": len(aggregate_alignment_df),
    "performance": len(performance_df),
}

assert actual_rows == expected_rows
assert magnitude_df["update_magnitude"].notna().all()
assert pairwise_df["cosine_similarity"].between(-1, 1).all()
assert aggregate_alignment_df["aggregate_alignment"].between(-1, 1).all()

pd.DataFrame({"expected": expected_rows, "observed": actual_rows})

## 5. Client heterogeneity

In [ ]:
digit_columns = [f"digit_{digit}" for digit in range(10)]
client_sizes = label_distribution_df.set_index("client")["samples"]
client_weights = client_sizes / client_sizes.sum()

display(label_distribution_df)

plt.figure(figsize=(11, 3.8))
sns.heatmap(
    label_distribution_df.set_index("client")[digit_columns],
    cmap="Blues",
    annot=True,
    fmt="g",
    cbar_kws={"label": "Training examples"},
)
plt.title("Saved Dirichlet partition: label counts per client")
plt.xlabel("Class")
plt.ylabel("Client")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "client_label_distribution.png", dpi=180)
plt.show()

pd.DataFrame({
    "samples": client_sizes,
    "fedavg_weight": client_weights,
})

Client 3 is the largest client, with 21,164 examples and a FedAvg weight of approximately 35.3%. Its data are concentrated mainly in digits 0, 1, 4, 5, and 7. The partition is therefore not pure label skew: it combines label concentration with substantial quantity imbalance.

## 6. Global performance

In [ ]:
ordered_performance = performance_df.sort_values(["mu", "round"])

performance_summary = (
    ordered_performance
    .groupby("mu")
    .agg(
        final_accuracy=("test_accuracy", "last"),
        best_accuracy=("test_accuracy", "max"),
        mean_last_5=("test_accuracy", lambda values: values.tail(5).mean()),
        final_loss=("test_loss", "last"),
    )
)

round_to_85 = (
    ordered_performance[ordered_performance["test_accuracy"] >= 0.85]
    .groupby("mu")["round"]
    .min()
)
performance_summary["round_to_85_percent"] = round_to_85
performance_summary

| μ | Final accuracy | Best accuracy | Mean accuracy, rounds 16–20 | Round to 85% | Final loss |
|---:|---:|---:|---:|---:|---:|
| 0.0 | 85.93% | 87.11% | 86.48% | 12 | 0.4162 |
| 0.1 | **86.75%** | **87.40%** | **87.07%** | **10** | **0.4034** |
| 1.0 | 85.20% | 85.99% | 85.36% | 17 | 0.4871 |

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.lineplot(
    data=performance_df,
    x="round",
    y="test_accuracy",
    hue="mu",
    marker="o",
    ax=axes[0],
)
axes[0].axhline(0.85, color="gray", linestyle="--", linewidth=1)
axes[0].set_title("Test accuracy by communication round")
axes[0].set_ylabel("Test accuracy")

sns.lineplot(
    data=performance_df,
    x="round",
    y="test_loss",
    hue="mu",
    marker="o",
    ax=axes[1],
)
axes[1].set_title("Test loss by communication round")
axes[1].set_ylabel("Cross-entropy loss")

plt.tight_layout()
plt.savefig(FIGURE_DIR / "performance_trajectories.png", dpi=180)
plt.show()

![Performance trajectories](../results/client_update_geometry/alpha_0p1/figures/performance_trajectories.png)

Moderate regularization (μ=0.1) provides the strongest overall trajectory. It reaches 85% accuracy two rounds earlier than FedAvg and has the highest mean accuracy over rounds 16–20. Strong regularization (μ=1) converges much more slowly, supporting H3.

## 7. Update magnitude

In [ ]:
magnitude_summary = magnitude_df.groupby("mu")["update_magnitude"].agg(
    ["mean", "std", "min", "max"]
)
fedavg_mean = magnitude_summary.loc[0.0, "mean"]
magnitude_summary["reduction_vs_fedavg_percent"] = (
    (fedavg_mean - magnitude_summary["mean"]) / fedavg_mean * 100
)
magnitude_summary

| μ | Mean update magnitude | Reduction relative to μ=0 |
|---:|---:|---:|
| 0.0 | 0.6541 | 0.0% |
| 0.1 | 0.5317 | 18.7% |
| 1.0 | 0.3295 | 49.6% |

In [ ]:
round_magnitude = (
    magnitude_df
    .groupby(["mu", "round"], as_index=False)["update_magnitude"]
    .mean()
)

plt.figure(figsize=(8.5, 4.8))
sns.lineplot(
    data=round_magnitude,
    x="round",
    y="update_magnitude",
    hue="mu",
    marker="o",
)
plt.title("Mean client-update magnitude")
plt.ylabel(r"Mean $\|\Delta_k\|_2$")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "mean_update_magnitude.png", dpi=180)
plt.show()

![Mean update magnitude](../results/client_update_geometry/alpha_0p1/figures/mean_update_magnitude.png)

H1 is supported in this run: stronger proximal regularization consistently reduces movement away from the round-start global model. However, the performance results show why magnitude cannot be interpreted as update quality. μ=1 produces the smallest updates but also the weakest late-round performance.

## 8. Pairwise directional alignment

In [ ]:
pairwise_summary = pairwise_df.groupby("mu")["cosine_similarity"].agg(
    ["mean", "std", "min", "max"]
)
pairwise_summary

| μ | Mean pairwise cosine | Standard deviation | Minimum | Maximum |
|---:|---:|---:|---:|---:|
| 0.0 | 0.0084 | 0.2354 | −0.3830 | 0.6790 |
| 0.1 | 0.0160 | 0.2451 | −0.4100 | 0.7016 |
| 1.0 | 0.0279 | 0.2754 | −0.5207 | 0.6611 |

In [ ]:
round_pairwise = (
    pairwise_df
    .groupby(["mu", "round"], as_index=False)["cosine_similarity"]
    .mean()
)

plt.figure(figsize=(8.5, 4.8))
sns.lineplot(
    data=round_pairwise,
    x="round",
    y="cosine_similarity",
    hue="mu",
    marker="o",
)
plt.axhline(0, color="black", linewidth=1)
plt.title("Mean pairwise client-update alignment")
plt.ylabel("Mean pairwise cosine similarity")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "mean_pairwise_alignment.png", dpi=180)
plt.show()

![Mean pairwise alignment](../results/client_update_geometry/alpha_0p1/figures/mean_pairwise_alignment.png)

In [ ]:
def pairwise_matrix(pairwise_df, mu, round_idx, clients):
    matrix = pd.DataFrame(np.eye(len(clients)), index=clients, columns=clients)
    selected = pairwise_df[
        pairwise_df["mu"].eq(mu) & pairwise_df["round"].eq(round_idx)
    ]

    for row in selected.itertuples(index=False):
        matrix.loc[row.client_i, row.client_j] = row.cosine_similarity
        matrix.loc[row.client_j, row.client_i] = row.cosine_similarity

    return matrix


clients = sorted(client_sizes.index)
fig, axes = plt.subplots(3, 3, figsize=(11, 10), sharex=True, sharey=True)

for row_idx, round_idx in enumerate([1, 10, 20]):
    for col_idx, mu in enumerate([0.0, 0.1, 1.0]):
        ax = axes[row_idx, col_idx]
        sns.heatmap(
            pairwise_matrix(pairwise_df, mu, round_idx, clients),
            ax=ax,
            cmap="coolwarm",
            vmin=-1,
            vmax=1,
            center=0,
            square=True,
            annot=True,
            fmt=".2f",
            cbar=col_idx == 2,
        )
        ax.set_title(f"round {round_idx}, μ={mu:g}")
        ax.set_xlabel("Client")
        ax.set_ylabel("Client")

plt.tight_layout()
plt.savefig(FIGURE_DIR / "pairwise_alignment_heatmaps.png", dpi=180)
plt.show()

![Pairwise alignment heatmaps](../results/client_update_geometry/alpha_0p1/figures/pairwise_alignment_heatmaps.png)

H2 receives only mixed support. Mean pairwise alignment is slightly higher when averaged across all rounds for larger μ, but the relationship changes over time. By round 20, all three conditions have negative mean pairwise alignment, and μ=1 is the most negative. We therefore should not claim that FedProx uniformly aligns client directions.

## 9. Ordinary aggregate alignment

In [ ]:
aggregate_summary = aggregate_alignment_df.groupby("mu")[
    "aggregate_alignment"
].agg(["mean", "std", "min", "max"])
aggregate_summary

In [ ]:
plt.figure(figsize=(9, 4.8))
sns.lineplot(
    data=aggregate_alignment_df,
    x="round",
    y="aggregate_alignment",
    hue="mu",
    style="client_id",
    markers=False,
    dashes=False,
    alpha=0.72,
)
plt.title("Client alignment with the self-inclusive aggregate update")
plt.ylabel(r"$\cos(\Delta_k, \bar\Delta)$")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "aggregate_alignment.png", dpi=180)
plt.show()

![Aggregate alignment](../results/client_update_geometry/alpha_0p1/figures/aggregate_alignment.png)

All ordinary aggregate-alignment means are positive. This does not imply that all clients agree with their peers: the reference vector contains the client being evaluated. The next section removes that mechanical self-contribution.

## 10. Leave-one-out aggregate alignment

In [ ]:
def reconstruct_alignment_diagnostics(
    magnitude_df,
    pairwise_df,
    aggregate_alignment_df,
    client_weights,
):
    '''Reconstruct ordinary and leave-one-out alignment from saved geometry.'''
    clients = sorted(client_weights.index)
    rows = []

    for (mu, round_idx), magnitude_group in magnitude_df.groupby(["mu", "round"]):
        magnitudes = magnitude_group.set_index("client_id")["update_magnitude"]
        cosine = pd.DataFrame(np.eye(len(clients)), index=clients, columns=clients)

        pair_group = pairwise_df[
            pairwise_df["mu"].eq(mu) & pairwise_df["round"].eq(round_idx)
        ]
        for pair in pair_group.itertuples(index=False):
            cosine.loc[pair.client_i, pair.client_j] = pair.cosine_similarity
            cosine.loc[pair.client_j, pair.client_i] = pair.cosine_similarity

        def weighted_norm(included_clients):
            squared_norm = 0.0
            for client_i in included_clients:
                for client_j in included_clients:
                    squared_norm += (
                        client_weights[client_i]
                        * client_weights[client_j]
                        * magnitudes[client_i]
                        * magnitudes[client_j]
                        * cosine.loc[client_i, client_j]
                    )
            return np.sqrt(max(squared_norm, 0.0))

        aggregate_norm = weighted_norm(clients)

        for client_id in clients:
            ordinary_numerator = sum(
                client_weights[peer_id]
                * magnitudes[peer_id]
                * cosine.loc[client_id, peer_id]
                for peer_id in clients
            )

            peers = [peer_id for peer_id in clients if peer_id != client_id]
            peer_norm = weighted_norm(peers)
            loo_numerator = sum(
                client_weights[peer_id]
                * magnitudes[peer_id]
                * cosine.loc[client_id, peer_id]
                for peer_id in peers
            )

            rows.append({
                "mu": mu,
                "round": round_idx,
                "client_id": client_id,
                "reconstructed_aggregate_alignment": (
                    ordinary_numerator / aggregate_norm
                ),
                "loo_aggregate_alignment": loo_numerator / peer_norm,
                "peer_aggregate_magnitude": peer_norm,
            })

    reconstructed_df = pd.DataFrame(rows)
    validation_df = reconstructed_df.merge(
        aggregate_alignment_df,
        on=["mu", "round", "client_id"],
    )
    max_error = (
        validation_df["reconstructed_aggregate_alignment"]
        - validation_df["aggregate_alignment"]
    ).abs().max()

    return reconstructed_df, max_error


loo_alignment_df, reconstruction_error = reconstruct_alignment_diagnostics(
    magnitude_df,
    pairwise_df,
    aggregate_alignment_df,
    client_weights,
)

assert reconstruction_error < 1e-5
loo_alignment_df.to_csv(
    ALPHA_DIR / "loo_alignment_E5_T20_seed42.csv",
    index=False,
)

print(f"Maximum reconstruction error: {reconstruction_error:.2e}")

In [ ]:
loo_summary = loo_alignment_df.groupby("mu")["loo_aggregate_alignment"].agg(
    ["mean", "std", "min", "max"]
)
loo_summary["negative_fraction"] = (
    loo_alignment_df
    .assign(negative=lambda frame: frame["loo_aggregate_alignment"] < 0)
    .groupby("mu")["negative"]
    .mean()
)
loo_summary

| μ | Mean ordinary alignment | Mean LOO alignment | Negative LOO fraction |
|---:|---:|---:|---:|
| 0.0 | 0.3871 | −0.0937 | 72% |
| 0.1 | 0.4056 | −0.0733 | 67% |
| 1.0 | 0.4207 | −0.0520 | 56% |

In [ ]:
round_loo = (
    loo_alignment_df
    .groupby(["mu", "round"], as_index=False)["loo_aggregate_alignment"]
    .mean()
)

plt.figure(figsize=(8.5, 4.8))
sns.lineplot(
    data=round_loo,
    x="round",
    y="loo_aggregate_alignment",
    hue="mu",
    marker="o",
)
plt.axhline(0, color="black", linewidth=1)
plt.title("Mean leave-one-out aggregate alignment")
plt.ylabel(r"Mean $\cos(\Delta_k, \bar\Delta_{-k})$")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "loo_alignment.png", dpi=180)
plt.show()

![Leave-one-out alignment](../results/client_update_geometry/alpha_0p1/figures/loo_alignment.png)

In [ ]:
client_alignment_comparison = (
    aggregate_alignment_df
    .groupby(["client_id", "mu"])["aggregate_alignment"]
    .mean()
    .rename("ordinary_alignment")
    .to_frame()
    .join(
        loo_alignment_df
        .groupby(["client_id", "mu"])["loo_aggregate_alignment"]
        .mean()
        .rename("loo_alignment")
    )
)
client_alignment_comparison

The contrast is substantial. At μ=0, mean ordinary aggregate alignment is approximately **0.387**, whereas mean leave-one-out alignment is approximately **−0.094**. Client 3 is the clearest case: its ordinary alignment is approximately **0.522**, but its peer-only alignment is approximately **−0.344**.

This does not prove that client 3 is harmful. It shows that a large client can strongly influence the server direction while moving against the coalition of the remaining clients.

## 11. Exploratory follow-up: scaling client 3's aggregation weight

This post-hoc intervention was motivated by client 3's large sample weight and persistent peer conflict. Its unnormalized FedAvg weight was multiplied by γ and then all client weights were renormalized:

\[
\widetilde p_k =
\frac{q_k}{\sum_jq_j},
\qquad
q_3=\gamma n_3,
\quad
q_k=n_k\ \text{for }k\ne3.
\]

γ=1 exactly reproduced the original μ=0 trajectory, validating the intervention implementation. The saved values below were reconstructed from the printed full-run log, so their precision is limited to four decimal places for loss and two decimal places for accuracy.

In [ ]:
scaling_summary = (
    scaling_performance_df
    .sort_values(["gamma", "round"])
    .groupby("gamma")
    .agg(
        final_accuracy=("test_accuracy", "last"),
        best_accuracy=("test_accuracy", "max"),
        mean_last_5=("test_accuracy", lambda values: values.tail(5).mean()),
        final_loss=("test_loss", "last"),
    )
)
scaling_summary

| γ | Final accuracy | Best accuracy | Mean accuracy, rounds 16–20 | Final loss |
|---:|---:|---:|---:|---:|
| 0.00 | 78.39% | 78.39% | 77.01% | 1.0086 |
| 0.25 | 85.05% | 85.05% | 84.57% | 0.4707 |
| 0.50 | 87.28% | 87.56% | 87.31% | 0.3933 |
| 0.75 | **87.29%** | **88.00%** | **87.55%** | **0.3887** |
| 1.00 | 85.93% | 87.11% | 86.48% | 0.4162 |

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.lineplot(
    data=scaling_performance_df,
    x="round",
    y="test_accuracy",
    hue="gamma",
    marker="o",
    palette="viridis",
    ax=axes[0],
)
axes[0].set_title("Client-3 scaling: test accuracy")
axes[0].set_ylabel("Test accuracy")

sns.lineplot(
    data=scaling_performance_df,
    x="round",
    y="test_loss",
    hue="gamma",
    marker="o",
    palette="viridis",
    ax=axes[1],
)
axes[1].set_title("Client-3 scaling: test loss")
axes[1].set_ylabel("Cross-entropy loss")

plt.tight_layout()
plt.savefig(FIGURE_DIR / "client3_scaling_trajectories.png", dpi=180)
plt.show()

![Client-3 scaling trajectories](../results/client_update_geometry/alpha_0p1/figures/client3_scaling_trajectories.png)

The relationship is non-monotonic. Complete removal (γ=0) performs poorly, while moderate attenuation (γ=0.5 or 0.75) improves the late-round trajectory relative to ordinary FedAvg (γ=1). This supports a narrow interpretation:

> In this particular seed and partition, client 3 contains useful information but its sample-size-proportional influence may be excessive under five local epochs.

Because the client and γ were selected after observing the geometry and test trajectory, this is an exploratory oracle intervention—not yet an adaptive algorithm or a general result.

## 12. Conclusions

### Hypothesis assessment

- **H1 — supported:** larger μ produces substantially smaller update norms.
- **H2 — mixed:** directional agreement changes with μ, but not uniformly across rounds; moderate FedProx cannot simply be described as an alignment mechanism.
- **H3 — supported in this run:** μ=1 over-constrains local movement and produces weaker convergence despite smaller updates.

### Main scientific interpretation

Moderate proximal regularization is associated with constrained client updates and improved communication-round convergence under the tested severe heterogeneity. Excessive regularization suppresses useful movement. Magnitude and direction respond differently, so neither update norm nor cosine similarity alone explains performance.

The leave-one-out analysis further shows that ordinary aggregate alignment can conceal peer disagreement, particularly for a large client. The client-scaling follow-up suggests that useful heterogeneity should be moderated carefully rather than removed automatically.

## 13. Limitations and next steps

1. Results currently use one seed, one model, one dataset, one saved partition, and five clients.
2. No confidence intervals or statistical-significance claims are justified.
3. α=0.1 creates both label concentration and quantity imbalance, so their effects are not isolated.
4. Cosine similarity is descriptive and does not establish why accuracy changes.
5. Leave-one-out alignment measures peer agreement, not client utility, fairness, or causal contribution.
6. The client-scaling experiment uses test performance to inspect fixed γ values and must not be presented as a deployable selection rule.

The highest-value robustness checks are additional seeds, (E=1) versus (E=5), per-class accuracy, and repeating the intervention for every client. FedAWA or a leave-one-out adaptive weighting method should only be added after these controls.

## Appendix A. Core diagnostic implementation

The cells below preserve the core implementation used to generate the saved CSV files. They are not rerun automatically when viewing this report. Set `RUN_FULL_EXPERIMENT=True` inside the repository only when a complete rerun is required.

In [ ]:
RUN_FULL_EXPERIMENT = False

if RUN_FULL_EXPERIMENT:
    import copy
    import random
    import sys
    from itertools import combinations

    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, Subset
    from torchvision import datasets, transforms

    # These modules are part of this repository.
    from src.data import create_client_loaders
    from src.diagnostics import (
        cosine_similarity_updates,
        model_to_vector,
        model_update_vector,
        update_magnitude,
    )
    from src.fedprox import client_update_fedprox
    from src.models import SimpleMLP
    from src.training import aggregate, evaluate

    print("Training dependencies imported.")
else:
    print("Using saved results; the 64-minute training run is not repeated.")

In [ ]:
def run_diagnostic_round(
    global_model,
    client_loaders,
    round_idx,
    local_epochs,
    learning_rate,
    loss_fn,
    device,
    mu,
):
    '''Train one FL round and measure all updates before aggregation.'''
    local_models = []
    client_sizes = []
    client_updates = {}

    global_vector = model_to_vector(global_model).clone()
    aggregate_update = torch.zeros_like(global_vector)
    total_client_size = 0
    magnitude_rows = []

    for client_id, client_loader in client_loaders.items():
        local_model = client_update_fedprox(
            global_model,
            client_loader,
            local_epochs,
            learning_rate,
            loss_fn,
            device,
            mu,
        )

        client_size = len(client_loader.dataset)
        delta = model_update_vector(local_model, global_model)

        local_models.append(local_model)
        client_sizes.append(client_size)
        client_updates[client_id] = delta
        total_client_size += client_size
        aggregate_update += client_size * delta

        magnitude_rows.append({
            "mu": mu,
            "round": round_idx,
            "local_epochs": local_epochs,
            "client_id": client_id,
            "update_magnitude": update_magnitude(delta).item(),
        })

    aggregate_update /= total_client_size

    aggregate_alignment_rows = [
        {
            "mu": mu,
            "round": round_idx,
            "local_epochs": local_epochs,
            "client_id": client_id,
            "aggregate_alignment": cosine_similarity_updates(
                delta,
                aggregate_update,
            ).item(),
        }
        for client_id, delta in client_updates.items()
    ]

    pairwise_rows = []
    for client_i, client_j in combinations(client_updates.keys(), 2):
        pairwise_rows.append({
            "mu": mu,
            "round": round_idx,
            "local_epochs": local_epochs,
            "client_i": client_i,
            "client_j": client_j,
            "cosine_similarity": cosine_similarity_updates(
                client_updates[client_i],
                client_updates[client_j],
            ).item(),
        })

    new_global_model = aggregate(global_model, local_models, client_sizes)

    # This catches ordering or weighting mistakes in the diagnostic calculation.
    server_update = model_to_vector(new_global_model) - global_vector
    assert torch.allclose(
        server_update,
        aggregate_update,
        rtol=1e-5,
        atol=1e-6,
    )

    return (
        new_global_model,
        magnitude_rows,
        pairwise_rows,
        aggregate_alignment_rows,
    )

In [ ]:
def run_diagnostic_experiment(
    mu_values,
    num_rounds,
    local_epochs,
    learning_rate,
    loss_fn,
    device,
    test_loader,
    client_loader_factory,
    seed,
    fresh_initial_model,
):
    '''Run an independent trajectory for each proximal coefficient.'''
    all_magnitude_rows = []
    all_pairwise_rows = []
    all_aggregate_alignment_rows = []
    performance_rows = []

    expected_num_pairs = 5 * 4 // 2

    for mu_value in mu_values:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

        global_model = fresh_initial_model()
        client_loaders = client_loader_factory()

        for round_idx in range(1, num_rounds + 1):
            (
                global_model,
                magnitude_rows,
                pairwise_rows,
                aggregate_alignment_rows,
            ) = run_diagnostic_round(
                global_model,
                client_loaders,
                round_idx,
                local_epochs,
                learning_rate,
                loss_fn,
                device,
                mu_value,
            )

            assert len(magnitude_rows) == 5
            assert len(pairwise_rows) == expected_num_pairs
            assert len(aggregate_alignment_rows) == 5

            test_loss, test_accuracy = evaluate(
                global_model,
                test_loader,
                loss_fn,
                device,
            )
            performance_rows.append({
                "mu": mu_value,
                "round": round_idx,
                "local_epochs": local_epochs,
                "test_loss": test_loss,
                "test_accuracy": test_accuracy,
            })

            all_magnitude_rows.extend(magnitude_rows)
            all_pairwise_rows.extend(pairwise_rows)
            all_aggregate_alignment_rows.extend(aggregate_alignment_rows)

    return (
        pd.DataFrame(all_magnitude_rows),
        pd.DataFrame(all_pairwise_rows),
        pd.DataFrame(all_aggregate_alignment_rows),
        pd.DataFrame(performance_rows),
    )

## Reproducibility note

The full experiment took approximately 64 minutes on Kaggle. The repository should retain:

- the exact initial checkpoint;
- the saved Dirichlet partition;
- the four raw diagnostic CSV files;
- the derived leave-one-out CSV;
- the reconstructed client-scaling performance log;
- this notebook with rendered outputs.

Keeping raw client- and pair-level values allows future analyses without rerunning local training.